# 1. Purpose and execution boundary

SCRUM-12 implements the approved Notebook 07 leakage-safe feature contract as an executable, bounded-memory Favorita pipeline. It validates formulas with deterministic fixtures, performs a bounded real-data smoke build, writes and reopens a smoke Parquet plus manifest, and records honest limitations.

The full training-origin schedule is not determined until SCRUM-13. Therefore this notebook does **not** create the canonical full model-ready artifact or claim the final training population is complete. It never loads the complete 125,497,040-row cleaned dataset into pandas or memory at once and never mutates the cleaned source.

## 2. Preserved SCRUM-9 to SCRUM-11 contracts

- Only observed source rows may become supervised target rows; absence does not prove zero demand.
- No store × item × date panel, synthetic zero-demand row, imputation, or silent densification is allowed.
- Negative and fractional `unit_sales` values remain valid target/history values.
- Forecast origin is end-of-day `t`; forecast dates are daily `t+1` through `t+16` under a direct horizon-aware design.
- Historical features use actual information at or before `t`; recursive prediction feedback is forbidden.
- Target-date promotion/holiday values require a planned/published-at-origin assumption. The cleaned artifact cannot prove that as-of availability, so the safe smoke configuration leaves these fields unknown.
- SCRUM-13 remains responsible for exact chronological folds and the final forecast-origin population.
- This notebook source, the production pipeline, and focused tests share the 16-horizon expectation. The saved outputs below record the controlled successful 16-day notebook execution and bounded smoke-artifact regeneration.

## 3. Imports, paths, and tested pipeline contract

In [1]:
from datetime import date
import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from pipelines.features.favorita_model_ready import (
    AUDIT_COLUMNS,
    FEATURE_DEFINITIONS,
    FORBIDDEN_MODEL_COLUMNS,
    FORECAST_HORIZONS,
    HOLIDAY_FEATURE_COLUMNS,
    INFERENCE_OUTPUT_COLUMNS,
    MODEL_FEATURE_COLUMNS,
    OUTPUT_ARROW_SCHEMA,
    SEMANTIC_TYPES,
    TARGET_COLUMN,
    TRAINING_OUTPUT_COLUMNS,
    FeatureBuildConfig,
    build_feature_manifest,
    materialize_feature_dataset,
    run_deterministic_fixture_validation,
    sha256_file,
    source_footer,
    validate_feature_artifact,
    write_json_atomic,
)

EXPECTED_FORECAST_HORIZONS = tuple(range(1, 17))
assert tuple(FORECAST_HORIZONS) == EXPECTED_FORECAST_HORIZONS, (
    "Notebook 08 and the production pipeline must share the 16-day contract"
)

SOURCE_PATH = Path("data/processed/favorita_cleaned/favorita_cleaned.parquet")
SOURCE_MANIFEST_PATH = Path("data/processed/favorita_cleaned/cleaning_manifest.json")
PLANNED_FULL_OUTPUT_PATH = Path(
    "data/processed/favorita_features/favorita_model_ready_features.parquet"
)
PLANNED_FULL_MANIFEST_PATH = Path(
    "data/processed/favorita_features/feature_manifest.json"
)
SMOKE_OUTPUT_PATH = Path(
    "data/processed/favorita_features/smoke/favorita_model_ready_features_smoke.parquet"
)
SMOKE_MANIFEST_PATH = Path(
    "data/processed/favorita_features/smoke/feature_manifest_smoke.json"
)

## 4. Validate the cleaned input without loading rows

The footer and SCRUM-10 manifest establish the source schema, row count, row groups, and validation status. A streaming SHA-256 plus file size and modification time are captured before the smoke build and rechecked afterward.

In [2]:
assert SOURCE_PATH.is_file(), f"Missing cleaned input: {SOURCE_PATH}"
assert SOURCE_MANIFEST_PATH.is_file(), f"Missing cleaning manifest: {SOURCE_MANIFEST_PATH}"

source_metadata_before = source_footer(SOURCE_PATH)
source_stat_before = SOURCE_PATH.stat()
source_sha256_before = sha256_file(SOURCE_PATH)
cleaning_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))

assert source_metadata_before["rows"] == 125_497_040
assert source_metadata_before["columns"] == 21
assert source_metadata_before["row_groups"] == 502
assert cleaning_manifest["validation"]["final_validation_passed"] is True
assert cleaning_manifest["validation"]["exact_ordered_row_content_preserved"] is True

source_summary = pd.DataFrame(
    [
        {"property": "rows", "value": source_metadata_before["rows"]},
        {"property": "columns", "value": source_metadata_before["columns"]},
        {"property": "row_groups", "value": source_metadata_before["row_groups"]},
        {"property": "file_size_bytes", "value": source_stat_before.st_size},
        {"property": "sha256", "value": source_sha256_before},
        {"property": "SCRUM-10 final validation", "value": True},
    ]
)
display(source_summary)

,property,value
0,rows,125497040
1,columns,21
2,row_groups,502
3,file_size_bytes,723900039
4,sha256,845b435622fc4fb31c5336fb1f6eda22195f64edba61a0...
5,SCRUM-10 final validation,True


## 5. Ordered output schema and semantic types

The training artifact contains audit fields, the observed `unit_sales` target, and exactly the approved feature groups. The inference schema is the identical ordered schema with only the target removed. Numeric-looking identifiers retain their original physical codes while the semantic contract requires categorical treatment.

In [3]:
schema_contract = pd.DataFrame(
    [
        {
            "position": position,
            "column": field.name,
            "arrow_type": str(field.type),
            "nullable": field.nullable,
            "semantic_type": SEMANTIC_TYPES[field.name],
            "in_inference_schema": field.name in INFERENCE_OUTPUT_COLUMNS,
        }
        for position, field in enumerate(OUTPUT_ARROW_SCHEMA)
    ]
)

assert tuple(OUTPUT_ARROW_SCHEMA.names) == TRAINING_OUTPUT_COLUMNS
assert tuple(column for column in TRAINING_OUTPUT_COLUMNS if column != TARGET_COLUMN) == INFERENCE_OUTPUT_COLUMNS
assert TARGET_COLUMN not in INFERENCE_OUTPUT_COLUMNS
assert not FORBIDDEN_MODEL_COLUMNS.intersection(MODEL_FEATURE_COLUMNS)
assert SEMANTIC_TYPES["forecast_horizon"] == "bounded integer horizon 1..16"
display(schema_contract)

,position,column,arrow_type,nullable,semantic_type,in_inference_schema
0,0,forecast_origin,timestamp[us],False,audit timestamp: end-of-day origin t,True
1,1,forecast_date,timestamp[us],False,audit timestamp: target date t+h,True
2,2,forecast_horizon,int8,False,bounded integer horizon 1..16,True
3,3,store_nbr,int16,False,categorical identifier (physical int16 code re...,True
4,4,item_nbr,int32,False,categorical identifier (physical int32 code re...,True
5,5,family,large_string,False,categorical,True
6,6,class,int16,False,categorical (physical int16 code retained),True
7,7,perishable,int8,False,binary,True
8,8,city,large_string,False,categorical,True
9,9,state,large_string,False,categorical,True


## 6. Exact historical formulas and null behavior

For forecast origin `t`, sales lags use exact prior calendar dates: `sales_lag_1 = t-1`, `sales_lag_7 = t-7`, `sales_lag_14 = t-14`, and `sales_lag_28 = t-28`. Sales rolling windows exclude the origin day and require every date in the previous complete calendar interval: 7 days `[t-7,t-1]`, 14 days `[t-14,t-1]`, and 28 days `[t-28,t-1]`. Any missing date makes the corresponding result null. Rolling standard deviations use sample `ddof=1`.

Transactions intentionally use different endpoints: `transactions_at_origin = t`; lags 7 and 14 use exact dates `t-7` and `t-14`; rolling means remain complete store-date windows `[t-6,t]` and `[t-13,t]`, including the origin day. Calendar fields are derived from `forecast_date`; `week_of_year` uses ISO-8601 week numbering. Oil 1-day and 7-day changes use exact dates. Seven-day rolling oil movement uses observed non-null prices in `[t-6,t]` without filling, while volatility is sample standard deviation of consecutive observed-price returns. Missing inputs, insufficient observations, or zero denominators produce null.


In [4]:
feature_definition_table = pd.DataFrame(
    FEATURE_DEFINITIONS.items(), columns=["feature", "definition"]
)
display(feature_definition_table)

,feature,definition
0,sales_lag_1,unit_sales on exact calendar date t-1
1,sales_lag_7,unit_sales on exact calendar date t-7
2,sales_lag_14,unit_sales on exact calendar date t-14
3,sales_lag_28,unit_sales on exact calendar date t-28
4,sales_rolling_mean_7,mean over complete observed calendar window [t...
5,sales_rolling_mean_14,mean over complete observed calendar window [t...
6,sales_rolling_mean_28,mean over complete observed calendar window [t...
7,sales_rolling_std_7,sample std (ddof=1) over complete observed cal...
8,sales_rolling_std_28,sample std (ddof=1) over complete observed cal...
9,transactions_at_origin,store transactions on exact calendar date t


## 7. Deterministic fixture and Parquet row-group boundary validation

The fixture independently calculates the approved sales dates `t-1`, `t-7`, `t-14`, and `t-28`; complete sales windows ending at `t-1`; transaction lags at `t-7` and `t-14`; and transaction means that still include `t`. It includes complete and sparse store-item histories, negative and fractional sales, nullable promotion/transactions/oil values, and an earthquake description used only for exclusion. A deliberately missing `t-7` sales row must null the exact lag and every affected complete sales window. Future sales and transaction values are mutated separately to prove that historical features do not change.

The fixture is also written with tiny row groups so each series crosses multiple boundaries. Direct in-memory results must exactly equal results reloaded from a source slice beginning at `t-28`.


In [5]:
fixture_validation_results = run_deterministic_fixture_validation()
assert fixture_validation_results
assert all(fixture_validation_results.values())
display(
    pd.DataFrame(
        fixture_validation_results.items(), columns=["fixture_check", "passed"]
    )
)

,fixture_check,passed
0,sales_lag_1_is_t_minus_1,True
1,sales_lag_7_is_t_minus_7,True
2,sales_lag_14_is_t_minus_14,True
3,sales_lag_28_is_t_minus_28,True
4,sales_windows_exclude_origin,True
5,complete_calendar_windows,True
6,missing_exact_date_remains_null,True
7,missing_date_invalidates_sales_windows,True
8,transactions_at_origin_is_t,True
9,transactions_lag_7_is_t_minus_7,True


## 8. Explicit bounded smoke configuration

The planned aligned smoke build uses one auditable origin (`2017-07-30`), Store 1, and at most the first 50 item codes with observed target-period rows. This origin leaves 16 observed training dates through `2017-08-15`. The source read is restricted to the required calendar interval `t-28` through `t+16`. Horizon expansion is limited to observed forecast-date rows; absent rows are never created.

Planned-at-origin promotion and holiday availability cannot be reconstructed from the cleaned artifact, so both assumptions are disabled and those future-known columns remain nullable unknown. The canonical full paths are declared but not written.

In [6]:
SMOKE_FORECAST_ORIGIN = date(2017, 7, 30)
smoke_config = FeatureBuildConfig(
    source_path=SOURCE_PATH,
    output_path=SMOKE_OUTPUT_PATH,
    manifest_path=SMOKE_MANIFEST_PATH,
    forecast_origins=(SMOKE_FORECAST_ORIGIN,),
    store_batches=((1,),),
    max_items_per_store=50,
    allow_assumed_future_promotion=False,
    allow_assumed_future_holidays=False,
    overwrite=True,
)

assert not PLANNED_FULL_OUTPUT_PATH.exists(), (
    "A canonical full artifact already exists; this bounded notebook must not overwrite it"
)
assert not PLANNED_FULL_MANIFEST_PATH.exists(), (
    "A canonical full manifest already exists; this bounded notebook must not overwrite it"
)

configuration_summary = pd.DataFrame(
    [
        {"setting": "forecast_origins", "value": [value.isoformat() for value in smoke_config.forecast_origins]},
        {"setting": "store_batches", "value": smoke_config.store_batches},
        {"setting": "max_items_per_store", "value": smoke_config.max_items_per_store},
        {"setting": "horizons", "value": list(EXPECTED_FORECAST_HORIZONS)},
        {"setting": "planned promotion assumption", "value": smoke_config.allow_assumed_future_promotion},
        {"setting": "published holiday assumption", "value": smoke_config.allow_assumed_future_holidays},
        {"setting": "full origin schedule complete", "value": False},
    ]
)
display(configuration_summary)

,setting,value
0,forecast_origins,[2017-07-30]
1,store_batches,"((1,),)"
2,max_items_per_store,50
3,horizons,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
4,planned promotion assumption,False
5,published holiday assumption,False
6,full origin schedule complete,False


## 9. Materialize the bounded real-data smoke artifact

PyArrow applies date/store filters and narrow-column projection. The pipeline processes the explicit origin/store batch, computes origin-bounded features, and writes incrementally through a temporary Parquet file. Footer/schema checks must pass before atomic replacement of the smoke output.

In [7]:
smoke_build_result = materialize_feature_dataset(smoke_config)
assert smoke_build_result["creation_status"] == "created"
assert smoke_build_result["artifact_validation"]["rows"] > 0
assert smoke_build_result["artifact_validation"]["rows"] == smoke_build_result["expected_observed_target_rows"]

display(
    pd.DataFrame(
        [
            {"property": "artifact", "value": SMOKE_OUTPUT_PATH.as_posix()},
            {"property": "creation_scope", "value": "bounded smoke only"},
            {"property": "batches_written", "value": smoke_build_result["batches_written"]},
            {"property": "rows", "value": smoke_build_result["artifact_validation"]["rows"]},
            {"property": "columns", "value": smoke_build_result["artifact_validation"]["columns"]},
        ]
    )
)

,property,value
0,artifact,data/processed/favorita_features/smoke/favorit...
1,creation_scope,bounded smoke only
2,batches_written,1
3,rows,548
4,columns,43


## 10. Parquet read-back, schema, coverage, and null evidence

Validation reopens every smoke row group, checks the declared Arrow schema and ordered columns, enforces the horizon/date equation and output grain, and reports date bounds, cardinalities, and all-column null counts.

In [8]:
artifact_validation = validate_feature_artifact(SMOKE_OUTPUT_PATH)
assert artifact_validation == smoke_build_result["artifact_validation"]
assert artifact_validation["columns"] == len(TRAINING_OUTPUT_COLUMNS)
assert artifact_validation["horizons"] == list(EXPECTED_FORECAST_HORIZONS)
assert artifact_validation["grain_duplicate_count"] == 0

artifact_summary = pd.DataFrame(
    [
        {"property": key, "value": value}
        for key, value in artifact_validation.items()
        if key not in {"schema", "null_counts"}
    ]
)
null_count_table = pd.DataFrame(
    artifact_validation["null_counts"].items(), columns=["column", "null_count"]
)
display(artifact_summary)
display(null_count_table)

,property,value
0,rows,548
1,columns,43
2,row_groups,1
3,forecast_date_min,2017-07-31
4,forecast_date_max,2017-08-15
5,store_cardinality,1
6,item_cardinality,50
7,horizons,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
8,grain_duplicate_count,0


,column,null_count
0,forecast_origin,0
1,forecast_date,0
2,forecast_horizon,0
3,store_nbr,0
4,item_nbr,0
5,family,0
6,class,0
7,perishable,0
8,city,0
9,state,0


## 11. Leakage, sparse-row, and train/inference validation

The validation below proves the smoke artifact contains no forbidden model fields, raw oil, future actual transactions, earthquake-derived features, or target in the inference schema. All future-known promotion/holiday values remain unknown under the disabled assumptions. Row equality with observed target rows establishes that no absent row was materialized as synthetic zero demand.

In [9]:
smoke_parquet = pq.ParquetFile(SMOKE_OUTPUT_PATH)
smoke_frame = smoke_parquet.read().to_pandas()

validation_results = {
    "fixture_formulas_passed": all(fixture_validation_results.values()),
    "corrected_sales_lags": all(
        fixture_validation_results[name]
        for name in (
            "sales_lag_1_is_t_minus_1",
            "sales_lag_7_is_t_minus_7",
            "sales_lag_14_is_t_minus_14",
            "sales_lag_28_is_t_minus_28",
        )
    ),
    "sales_windows_exclude_origin": fixture_validation_results[
        "sales_windows_exclude_origin"
    ],
    "missing_date_invalidates_sales_windows": fixture_validation_results[
        "missing_date_invalidates_sales_windows"
    ],
    "corrected_transaction_lags": (
        fixture_validation_results["transactions_lag_7_is_t_minus_7"]
        and fixture_validation_results["transactions_lag_14_is_t_minus_14"]
    ),
    "transaction_windows_include_origin": fixture_validation_results[
        "transaction_windows_include_origin"
    ],
    "future_actuals_do_not_affect_historical_features": fixture_validation_results[
        "future_actuals_do_not_affect_historical_features"
    ],
    "cross_row_group_correctness": fixture_validation_results[
        "cross_row_group_equivalence"
    ],
    "ordered_arrow_schema_matches": smoke_parquet.schema_arrow.equals(OUTPUT_ARROW_SCHEMA),
    "horizon_restricted_to_1_16": set(smoke_frame["forecast_horizon"]) == set(EXPECTED_FORECAST_HORIZONS),
    "forecast_date_equation": (
        smoke_frame["forecast_date"]
        == smoke_frame["forecast_origin"]
        + pd.to_timedelta(smoke_frame["forecast_horizon"], unit="D")
    ).all(),
    "forbidden_model_columns_absent": not bool(FORBIDDEN_MODEL_COLUMNS.intersection(MODEL_FEATURE_COLUMNS)),
    "raw_dcoilwtico_absent": "dcoilwtico" not in smoke_frame.columns,
    "future_actual_transactions_absent": "transactions" not in smoke_frame.columns,
    "future_actual_sales_absent_from_features": TARGET_COLUMN not in MODEL_FEATURE_COLUMNS,
    "earthquake_features_absent": not any("earthquake" in column for column in smoke_frame.columns),
    "holiday_description_absent": "holiday_description" not in smoke_frame.columns,
    "no_synthetic_target_rows": len(smoke_frame) == smoke_build_result["expected_observed_target_rows"],
    "output_grain_unique": artifact_validation["grain_duplicate_count"] == 0,
    "categorical_semantics_declared": all(
        "categorical" in SEMANTIC_TYPES[column]
        for column in ("store_nbr", "item_nbr", "family", "class", "city", "state", "store_type", "cluster")
    ),
    "train_inference_ordered_schema_parity": tuple(
        column for column in TRAINING_OUTPUT_COLUMNS if column != TARGET_COLUMN
    ) == INFERENCE_OUTPUT_COLUMNS,
    "target_absent_from_inference": TARGET_COLUMN not in INFERENCE_OUTPUT_COLUMNS,
    "future_promotion_unknown": smoke_frame["onpromotion"].isna().all(),
    "future_holiday_fields_unknown": smoke_frame[list(HOLIDAY_FEATURE_COLUMNS)].isna().all().all(),
}
assert all(validation_results.values()), validation_results
display(pd.DataFrame(validation_results.items(), columns=["validation", "passed"]))

,validation,passed
0,fixture_formulas_passed,True
1,corrected_sales_lags,True
2,sales_windows_exclude_origin,True
3,missing_date_invalidates_sales_windows,True
4,corrected_transaction_lags,True
5,transaction_windows_include_origin,True
6,future_actuals_do_not_affect_historical_features,True
7,cross_row_group_correctness,True
8,ordered_arrow_schema_matches,True
9,horizon_restricted_to_1_16,True


## 12. Confirm cleaned-source immutability

The smoke build may read filtered source rows but must not change the cleaned Parquet. Size, modification time, footer facts, and streaming SHA-256 are compared before and after processing.

In [10]:
source_metadata_after = source_footer(SOURCE_PATH)
source_stat_after = SOURCE_PATH.stat()
source_sha256_after = sha256_file(SOURCE_PATH)

input_unchanged = (
    source_metadata_after == source_metadata_before
    and source_stat_after.st_size == source_stat_before.st_size
    and source_stat_after.st_mtime_ns == source_stat_before.st_mtime_ns
    and source_sha256_after == source_sha256_before
)
assert input_unchanged
validation_results["cleaned_source_unchanged"] = input_unchanged

display(
    pd.DataFrame(
        [
            {"check": "footer/schema unchanged", "passed": source_metadata_after == source_metadata_before},
            {"check": "file size unchanged", "passed": source_stat_after.st_size == source_stat_before.st_size},
            {"check": "mtime unchanged", "passed": source_stat_after.st_mtime_ns == source_stat_before.st_mtime_ns},
            {"check": "SHA-256 unchanged", "passed": source_sha256_after == source_sha256_before},
        ]
    )
)

,check,passed
0,footer/schema unchanged,True
1,file size unchanged,True
2,mtime unchanged,True
3,SHA-256 unchanged,True


## 13. Create and validate the bounded smoke manifest

The manifest records source evidence, output schema and statistics, semantic types, formulas, explicit origin/store/item configuration, null and sparse-data rules, forbidden-feature assertions, processing method, validation results, reproducibility versions, and unresolved limitations. It is written atomically and reopened as JSON.

In [11]:
limitations = [
    "This is a bounded smoke artifact, not the complete model-ready training population.",
    "SCRUM-13 must define the final chronological forecast-origin schedule and hold-out policy.",
    "The cleaned artifact has no publication-time evidence for planned promotions or holidays; both future-known groups remain unknown in this smoke build.",
    "No encoders or vocabularies are fitted in SCRUM-12; categorical codes and semantic types are retained for later training-only fitting.",
]

feature_manifest = build_feature_manifest(
    config=smoke_config,
    source_metadata=source_metadata_before,
    source_sha256=source_sha256_before,
    build_result=smoke_build_result,
    validation_results=validation_results,
    creation_scope="bounded_real_data_smoke_only",
    limitations=limitations,
)
feature_manifest["planned_full_artifact"] = {
    "output_path": PLANNED_FULL_OUTPUT_PATH.as_posix(),
    "manifest_path": PLANNED_FULL_MANIFEST_PATH.as_posix(),
    "created": False,
}

write_json_atomic(feature_manifest, SMOKE_MANIFEST_PATH, overwrite=True)
manifest_read_back = json.loads(SMOKE_MANIFEST_PATH.read_text(encoding="utf-8"))

assert manifest_read_back == feature_manifest
assert manifest_read_back["artifact"]["creation_scope"] == "bounded_real_data_smoke_only"
assert manifest_read_back["output"]["rows"] == artifact_validation["rows"]
assert manifest_read_back["output"]["columns"] == artifact_validation["columns"]
assert manifest_read_back["planned_full_artifact"]["created"] is False
assert manifest_read_back["forbidden_feature_assertions"]["target_absent_from_inference"] is True

print(f"Smoke feature manifest read-back: PASS — {SMOKE_MANIFEST_PATH}")

Smoke feature manifest read-back: PASS — data/processed/favorita_features/smoke/feature_manifest_smoke.json


## 14. Final SCRUM-12 execution summary

This notebook validates the reusable leakage-safe pipeline and creates only a bounded real-data smoke Parquet plus smoke manifest. It does not create the canonical full feature artifact because the authoritative full forecast-origin schedule remains unresolved until SCRUM-13. The result is implementation evidence, not a claim that the final training population is complete.

In [12]:
final_status = pd.DataFrame(
    [
        {"status_item": "deterministic formula tests", "value": "PASS"},
        {"status_item": "cross-row-group correctness", "value": "PASS"},
        {"status_item": "bounded real-data smoke artifact", "value": SMOKE_OUTPUT_PATH.as_posix()},
        {"status_item": "smoke manifest", "value": SMOKE_MANIFEST_PATH.as_posix()},
        {"status_item": "smoke rows", "value": artifact_validation["rows"]},
        {"status_item": "smoke columns", "value": artifact_validation["columns"]},
        {"status_item": "forecast-date coverage", "value": (artifact_validation["forecast_date_min"], artifact_validation["forecast_date_max"])},
        {"status_item": "full canonical artifact created", "value": False},
        {"status_item": "complete training population claimed", "value": False},
        {"status_item": "cleaned source changed", "value": False},
        {"status_item": "model trained", "value": False},
        {"status_item": "SCRUM-13 backtesting implemented", "value": False},
    ]
)
display(final_status)

,status_item,value
0,deterministic formula tests,PASS
1,cross-row-group correctness,PASS
2,bounded real-data smoke artifact,data/processed/favorita_features/smoke/favorit...
3,smoke manifest,data/processed/favorita_features/smoke/feature...
4,smoke rows,548
5,smoke columns,43
6,forecast-date coverage,"(2017-07-31, 2017-08-15)"
7,full canonical artifact created,False
8,complete training population claimed,False
9,cleaned source changed,False
